In [ ]:
%matplotlib inline
%reload_ext autoreload
%autoreload 2

# 6.11 — Prior-state dynamics: access vs maintenance

The DDM says PD fails to bias decisions. GLM-HMM asks a sharper question: is the prior-using
state **harder to enter** (access) or **harder to stay in** (maintenance)? From each session's
transition matrix (aligned to the shared basis): **entry** = inflow into the prior state,
**dwell** = its self-transition (maintenance). Prediction (and preliminary result): the PD
deficit is **maintenance** — tremor patients have low prior-state dwell OFF medication,
restored ON — a mechanistic claim the DDM can't make.

In [ ]:
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import wilcoxon
from scipy.optimize import linear_sum_assignment

from imports import *
from config import dir_config
from src.shared_glm_hmm.grouping import session_metadata

In [ ]:
import json

processed_dir = Path(dir_config.data.processed)
shared_dir = processed_dir / "shared_glm_hmm"
# Track the model finalized in 6.00 (selected_model.json); fall back to XY__tgt if absent.
_sel = shared_dir / "selected_model.json"
SELECTED = json.loads(_sel.read_text())["config"] if _sel.exists() else "XY__tgt"
CONFIG = SELECTED

bundle = pickle.load(open(shared_dir / CONFIG / "all_subjects_final.pkl", "rb"))
pooled = bundle["model"]["pooled"]
smodels = bundle["model"]["models"]
feats = bundle["config"]["model_features"]
K = bundle["best_k"]
data = bundle["data"]
ci = feats.index("color")
ref_w = -pooled.observations.params[:, 0, :]
prior_state = int(np.argmax(ref_w[:, ci]))
# state names as used in 6.02 (S-keyed); sname() falls back to S{k} if K differs.
state_names = {"S0": "Positive biased", "S1": "Negative biased", "S2": "Unbiased", "S3": "Lapse"}
sname = lambda k: state_names.get(f"S{k}", f"S{k}")
sess_md = session_metadata(pd.read_csv(processed_dir / "processed_metadata_all_data_accu_60.csv")).set_index("session_id")
print(f"CONFIG={CONFIG}  K={K}  prior_state=S{prior_state} ({sname(prior_state)})")

## 1. Per-session transition dynamics (aligned to the shared basis)

In [ ]:
def align_perm(w, ref):
    cost = np.array([[np.sum((w[i]-ref[j])**2) for j in range(len(ref))] for i in range(len(w))])
    _, col = linear_sum_assignment(cost); inv = np.empty_like(col); inv[col] = np.arange(len(col)); return inv

Ts = {}   # per-session aligned KxK transition matrix
rows = []
for sid, m in smodels.items():
    perm = align_perm(-m.observations.params[:, 0, :], ref_w)
    T = m.transitions.transition_matrix[np.ix_(perm, perm)]
    Ts[sid] = T
    P = prior_state
    df = data[sid]
    obs = df["choices"].values.astype(int).reshape(-1, 1); inp = df[feats].values.astype(float)
    msk = df["mask"].values.astype(bool).reshape(-1, 1)
    post = m.expected_states(data=obs, input=inp, mask=msk)[0][:, perm]; v = msk[:, 0]
    rows.append(dict(session_id=sid,
                     dwell=T[P, P],
                     dwell_time=1.0 / (1.0 - T[P, P] + 1e-9),
                     entry=np.mean([T[k, P] for k in range(K) if k != P]),
                     switch=(np.diff(np.argmax(post[v], 1)) != 0).mean()))
dyn = pd.DataFrame(rows).set_index("session_id").join(sess_md)
print("prior-state dynamics:", dyn.shape)
dyn.groupby(["subtype", "medication"], observed=True)[["dwell", "entry", "switch"]].agg(["mean", "sem", "count"]).round(3)

## 2. Within-subject OFF→ON: access (entry) vs maintenance (dwell)

In [ ]:
pd_dyn = dyn[dyn["is_pd"] == 1].copy()
def paired(metric, sub):
    pv = (pd_dyn.pivot_table(index=["subject_id", "subtype"], columns="medication",
                             values=metric, observed=True).dropna(subset=["off", "on"]).reset_index())
    d = pv if sub is None else pv[pv["subtype"] == sub]
    if len(d) < 3: return dict(metric=metric, group=sub or "all_PD", n=len(d), off=np.nan, on=np.nan, delta=np.nan, p=np.nan)
    _, p = wilcoxon(d["off"], d["on"])
    return dict(metric=metric, group=sub or "all_PD", n=len(d), off=d["off"].mean(), on=d["on"].mean(),
                delta=(d["on"]-d["off"]).mean(), p=p)
res = pd.DataFrame([paired(mtr, sub) for mtr in ["dwell", "entry", "switch"] for sub in [None, "tremor", "bradykinetic"]])
print("Paired Wilcoxon OFF vs ON (dwell=maintenance, entry=access):")
res.set_index(["metric", "group"]).round(4)

## 3. Maintenance (dwell) OFF→ON, per subtype, vs HC reference

In [ ]:
hc = dyn[dyn["subtype"] == "HC"]["dwell"]; hc_m, hc_s = hc.mean(), hc.sem()
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)
for ax, sub in zip(axes, ["tremor", "bradykinetic"]):
    pv = (pd_dyn.pivot_table(index="subject_id", columns="medication", values="dwell", observed=True)
          .dropna(subset=["off", "on"]))
    sub_subj = pd_dyn[pd_dyn["subtype"] == sub]["subject_id"].unique()
    d = pv.loc[pv.index.isin(sub_subj)]
    for _, r in d.iterrows():
        ax.plot([0, 1], [r["off"], r["on"]], "-o", color="gray", alpha=0.45, lw=1, ms=4)
    ax.errorbar([0, 1], [d["off"].mean(), d["on"].mean()], yerr=[d["off"].sem(), d["on"].sem()],
                color="tab:red", lw=3, marker="o", ms=10, capsize=6, zorder=5)
    ax.axhspan(hc_m - hc_s, hc_m + hc_s, color="tab:green", alpha=0.15)
    ax.axhline(hc_m, color="tab:green", ls="--", lw=1.5, label="HC")
    ax.set_xticks([0, 1]); ax.set_xticklabels(["OFF", "ON"]); ax.set_xlim(-0.3, 1.3)
    ax.set_title(f"{sub}  (n={len(d)})"); ax.legend(fontsize=9)
axes[0].set_ylabel(f"prior-state dwell  (self-transition)")
fig.suptitle(f"{CONFIG} — prior-state maintenance OFF→ON", fontsize=14)
plt.tight_layout(); plt.show()

## 4. Group-mean transition matrices

In [ ]:
groups = [("HC","none"), ("tremor","off"), ("tremor","on"), ("bradykinetic","off"), ("bradykinetic","on")]
groups = [gp for gp in groups if ((dyn["subtype"]==gp[0]) & (dyn["medication"]==gp[1])).any()]
labels = [state_names[k] for k in range(K)]
fig, axes = plt.subplots(1, len(groups), figsize=(4.2*len(groups), 4), squeeze=False)
for ax, gp in zip(axes[0], groups):
    sids = dyn[(dyn["subtype"]==gp[0]) & (dyn["medication"]==gp[1])].index
    Tm = np.mean([Ts[s] for s in sids], axis=0)
    im = ax.imshow(Tm, vmin=0, vmax=1, cmap="viridis")
    ax.set_xticks(range(K)); ax.set_yticks(range(K))
    ax.set_xticklabels([f"S{k}" for k in range(K)]); ax.set_yticklabels([f"S{k}" for k in range(K)])
    ax.set_title(f"{gp[0]} {gp[1]}\n(n={len(sids)})", fontsize=10)
    # outline the prior state self-transition
    ax.add_patch(plt.Rectangle((prior_state-0.5, prior_state-0.5), 1, 1, fill=False, ec="red", lw=2))
fig.colorbar(im, ax=axes[0], fraction=0.025)
fig.suptitle(f"{CONFIG} — mean transition matrix (rows=from, cols=to; red box = prior-state dwell)", fontsize=13, y=1.04)
plt.show()

## 5. Read-out

If the OFF→ON change is in **dwell** (not **entry**), the PD prior-integration deficit is one
of **maintenance, not access** — patients can enter the prior-using state but can't sustain
it, and medication restores the sustain (tremor). This is the mechanistic dissociation the
DDM starting-point/drift result can't deliver, and it explains why explicit instruction does
not rescue PD priors: instruction can trigger a transition it cannot keep stable.